### Import Dependencies.

In [1]:
from datasets import load_dataset
import torch
from transformers import AutoTokenizer , AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments , Trainer
from transformers import pipeline

### Load Model and Tokenizer

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Load Dataset[SamSum]

In [4]:
dataset = load_dataset("knkarthick/samsum")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

### Tokenize the data

In [6]:
def tokenize_content(batch_data): 
    # step-1: Get the rows values for a batch
    dialogue = batch_data['dialogue'] 
    summary = batch_data['summary']
    # step-2: loop through each batch content to format the data
    # t5 needs the task at the begging as a token like summarize
    inputs = ["summarize: " + d if d else "summarize: " for d in dialogue]
    targets = [s if s else "" for s in summary]
    
    # step-3: tokenize both inputs and targets
    inputs_encoding = tokenizer(
        inputs, max_length = 1024, truncation = True, padding = 'max_length' 
    )
    with tokenizer.as_target_tokenizer():
        target_encodings = tokenizer(
            targets , max_length = 128, truncation = True, padding = 'max_length' 
        )
    
    return { 
       'input_ids': inputs_encoding['input_ids'],
       'attention_mask': inputs_encoding['attention_mask'],
       'labels': target_encodings['input_ids']
    }

In [7]:
tokenized_dataset = dataset.map( 
   tokenize_content, 
   batched = True
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\transformers\tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [8]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

### Setup Data Collator
 - make batches for training and after completion remove the batch from cache.

In [9]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model
)

### Define Model Arguments

In [10]:
training_args = TrainingArguments(
    output_dir = '../t5-samsum-model',
    learning_rate = 2e-5,
    num_train_epochs = 2,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    warmup_steps = 500,
    weight_decay = 0.01,
    logging_steps = 10,
    eval_steps = 500,
    save_steps = 1e6,
    gradient_accumulation_steps = 16,
    report_to = None,
    eval_strategy = 'epoch'
)

### Define Trainer

In [11]:
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation']
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,8.442200,7.351338
2,1.324300,0.703903


TrainOutput(global_step=232, training_loss=7.014977983359633, metrics={'train_runtime': 1179.3537, 'train_samples_per_second': 24.981, 'train_steps_per_second': 0.197, 'total_flos': 7974880309936128.0, 'train_loss': 7.014977983359633, 'epoch': 2.0})

### Save Model and Tokenizer

In [13]:
model.save_pretrained('t5-samsum-finetuined-model')
tokenizer.save_pretrained('t5-samsum-tokenizer')

('t5-samsum-tokenizer\\tokenizer_config.json',
 't5-samsum-tokenizer\\special_tokens_map.json',
 't5-samsum-tokenizer\\spiece.model',
 't5-samsum-tokenizer\\added_tokens.json',
 't5-samsum-tokenizer\\tokenizer.json')

### Load the model and tokenizer for inference

In [14]:
tuned_model = AutoModelForSeq2SeqLM.from_pretrained(
    't5-samsum-finetuined-model'
).to(device)

In [15]:
tokenizer_new = AutoTokenizer.from_pretrained(
    't5-samsum-tokenizer'
)

In [16]:
summarizer = pipeline(
    task = 'summarization',
    model = tuned_model,
    tokenizer = tokenizer_new
)

Device set to use cuda:0


In [17]:
test_context = """ 
Alice: Did you finish the project report?
Bob: Not yet. I'm still working on the results section.
Alice: The deadline is tomorrow morning.
Bob: I know. I'll finish it tonight and send it to you.
Alice: Please do. I need to review it before submission.
Bob: Sure. I'll email you by midnight.
Alice: Thanks.
"""

In [22]:
from IPython.display import Markdown, display
result = summarizer(test_context , do_sample = False , max_length = 40)
display(
    Markdown(f"Summary {result[0]['summary_text']}")
)

Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary "I know. I'll finish it tonight and send it to you," she says . she says the deadline is tomorrow morning .

In [23]:
result

[{'summary_text': '"I know. I\'ll finish it tonight and send it to you," she says . she says the deadline is tomorrow morning .'}]